In the previous notebook I explored the Rossmann sales data.
In this notebook I focus on feature engineering, model training and evaluation for the sales forecasting task.

In [8]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as pt 
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit

In [4]:
df_sales = pd.read_csv('../data/train.csv', low_memory=False)
df_store = pd.read_csv('../data/store.csv', low_memory=False)

In [6]:
def merge_store(df_store : pd.DataFrame, df_sales : pd.DataFrame) -> pd.DataFrame:
    df_store_copy = df_store.copy()
    df_sales_copy = df_sales.copy()
    df_sales_copy['Date'] = pd.to_datetime(df_sales_copy['Date'])
    df_full = pd.merge(df_store_copy, df_sales_copy, on='Store', how='left')
    return df_full 

df_full = merge_store(df_store, df_sales)

In [7]:
df_full.info()
df_full.head()

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 18 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   Store                      1017209 non-null  int64         
 1   StoreType                  1017209 non-null  str           
 2   Assortment                 1017209 non-null  str           
 3   CompetitionDistance        1014567 non-null  float64       
 4   CompetitionOpenSinceMonth  693861 non-null   float64       
 5   CompetitionOpenSinceYear   693861 non-null   float64       
 6   Promo2                     1017209 non-null  int64         
 7   Promo2SinceWeek            509178 non-null   float64       
 8   Promo2SinceYear            509178 non-null   float64       
 9   PromoInterval              509178 non-null   str           
 10  DayOfWeek                  1017209 non-null  int64         
 11  Date                       1017209 non-null  dat

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,5,2015-07-31,5263,555,1,1,0,1
1,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,4,2015-07-30,5020,546,1,1,0,1
2,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,3,2015-07-29,4782,523,1,1,0,1
3,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,2,2015-07-28,5011,560,1,1,0,1
4,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,1,2015-07-27,6102,612,1,1,0,1


In [14]:
df_full = df_full.sort_values('Date')

y = df_full['Sales'].copy() 
X = df_full.drop(columns=['Sales'])

In [17]:
X['Date'].head(3000)

1017208   2013-01-01
679363    2013-01-01
155193    2013-01-01
632403    2013-01-01
361623    2013-01-01
             ...    
422711    2013-01-03
65797     2013-01-03
147081    2013-01-03
129161    2013-01-03
907662    2013-01-03
Name: Date, Length: 3000, dtype: datetime64[us]

In [18]:
X.info()

<class 'pandas.DataFrame'>
Index: 1017209 entries, 1017208 to 0
Data columns (total 17 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   Store                      1017209 non-null  int64         
 1   StoreType                  1017209 non-null  str           
 2   Assortment                 1017209 non-null  str           
 3   CompetitionDistance        1014567 non-null  float64       
 4   CompetitionOpenSinceMonth  693861 non-null   float64       
 5   CompetitionOpenSinceYear   693861 non-null   float64       
 6   Promo2                     1017209 non-null  int64         
 7   Promo2SinceWeek            509178 non-null   float64       
 8   Promo2SinceYear            509178 non-null   float64       
 9   PromoInterval              509178 non-null   str           
 10  DayOfWeek                  1017209 non-null  int64         
 11  Date                       1017209 non-null  datetime

In [19]:
tscv = TimeSeriesSplit(n_splits=3)

for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]